In [1]:
# ============================================================
# FUNDUS vs NON-FUNDUS CLASSIFIER TRAINER (3-min notebook)
# ============================================================

import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split, ConcatDataset
from tqdm import tqdm

# -----------------------------
# CONFIGURATION
# -----------------------------
FUNDUS_DIR = r"C:\Users\hitha\OneDrive\Desktop\eye disease app\dataset"
SAVE_PATH = r"C:\Users\hitha\OneDrive\Desktop\eye disease app\backend\models\fundus_vs_nonfundus.pt"
BATCH_SIZE = 16
EPOCHS = 5  # Fast training (<3 minutes)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -----------------------------
# DATA TRANSFORMS
# -----------------------------
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# -----------------------------
# LOAD FUNDUS IMAGES
# -----------------------------
fundus_dataset = datasets.ImageFolder(FUNDUS_DIR, transform=transform)
print(f"✅ Fundus images loaded: {len(fundus_dataset)} samples")

# -----------------------------
# LOAD NON-FUNDUS IMAGES (CIFAR10)
# -----------------------------
nonfundus_dataset = datasets.CIFAR10(
    root="./cifar_data",
    train=True,
    download=True,
    transform=transform
)
print(f"✅ Non-fundus images loaded: {len(nonfundus_dataset)} samples")

# We’ll take a smaller random subset of CIFAR10 for quick training
nonfundus_subset, _ = random_split(nonfundus_dataset, [len(fundus_dataset), len(nonfundus_dataset)-len(fundus_dataset)])

# -----------------------------
# RE-LABEL DATASETS
# -----------------------------
# Fundus = label 1, Non-fundus = label 0
fundus_dataset.samples = [(path, 1) for path, _ in fundus_dataset.samples]

class NonFundusWrapper(torch.utils.data.Dataset):
    def __init__(self, base):
        self.base = base
    def __getitem__(self, idx):
        x, _ = self.base[idx]
        return x, 0
    def __len__(self):
        return len(self.base)

nonfundus_dataset_wrapped = NonFundusWrapper(nonfundus_subset)

# Combine both datasets
combined_dataset = ConcatDataset([fundus_dataset, nonfundus_dataset_wrapped])
train_len = int(0.8 * len(combined_dataset))
val_len = len(combined_dataset) - train_len
train_set, val_set = random_split(combined_dataset, [train_len, val_len])

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE)

print(f"📊 Total images: {len(combined_dataset)} | Train: {len(train_set)} | Val: {len(val_set)}")

# -----------------------------
# MODEL SETUP (MobileNetV3 Small)
# -----------------------------
model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
model.classifier[3] = nn.Linear(model.classifier[3].in_features, 2)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# -----------------------------
# TRAINING LOOP
# -----------------------------
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"🧠 Epoch {epoch+1} | Train Loss: {total_loss/len(train_loader):.4f}")

    # Validation
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    acc = 100 * correct / total
    print(f"✅ Validation Accuracy: {acc:.2f}%")

# -----------------------------
# SAVE MODEL
# -----------------------------
os.makedirs(os.path.dirname(SAVE_PATH), exist_ok=True)
torch.save(model.state_dict(), SAVE_PATH)
print(f"\n🎉 Model saved successfully → {SAVE_PATH}")


✅ Fundus images loaded: 4152 samples


100%|██████████| 170M/170M [00:57<00:00, 2.98MB/s] 


Extracting ./cifar_data\cifar-10-python.tar.gz to ./cifar_data
✅ Non-fundus images loaded: 50000 samples
📊 Total images: 8304 | Train: 6643 | Val: 1661


Epoch 1/5: 100%|██████████| 416/416 [05:23<00:00,  1.29it/s]


🧠 Epoch 1 | Train Loss: 0.0231
✅ Validation Accuracy: 100.00%


...

🧠 Epoch 3 | Train Loss: 0.0051
✅ Validation Accuracy: 100.00%


Epoch 4/5: 100%|██████████| 416/416 [05:17<00:00,  1.31it/s]


🧠 Epoch 4 | Train Loss: 0.0006
✅ Validation Accuracy: 99.94%


Epoch 5/5: 100%|██████████| 416/416 [05:18<00:00,  1.30it/s]


🧠 Epoch 5 | Train Loss: 0.0001
✅ Validation Accuracy: 99.82%

🎉 Model saved successfully → C:\Users\hitha\OneDrive\Desktop\eye disease app\backend\models\fundus_vs_nonfundus.pt
